# 07 — Content Analysis (SVD / Topic Modeling)

**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`
**Output:** `../outputs/tables/svd_component_exemplars.csv`, `../outputs/tables/svd_explained_variance.csv`

**Description:**
- SVD / PCA on embeddings for interpretable content dimensions
- Exemplar texts per component (high and low extremes)
- Within-person extreme days: what does a person's unusual text look like?
- Matches original cells 20, 22, 29, 43

Note: BERTopic requires additional packages (umap-learn, hdbscan, bertopic).
If available, the final cell runs topic modeling. If not, it is skipped gracefully.

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_DIR = os.path.join("..", "outputs", "tables")
FIG_DIR = os.path.join("..", "outputs", "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
TEXT_COL = "pm_day_text"
CRISIS_COL = "crisis_PM_from_full"

N_COMPONENTS = 20
N_EXEMPLARS = 5
MIN_WORDS = 6
RANDOM_SEED = 7

In [ ]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)
assert X_text.shape[0] == len(pm_day)

pm_day["_wc"] = pm_day[TEXT_COL].fillna("").astype(str).str.split().map(len)
print("Loaded data:", pm_day.shape)
print("Loaded embeddings:", X_text.shape)

In [ ]:
# =========================
# HELPERS
# =========================
def l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)


def anonymize_text(s):
    s = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "[EMAIL]", s)
    s = re.sub(r"https?://\S+|www\.\S+", "[URL]", s)
    s = re.sub(r"\d", "X", s)
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
# =========================
# SVD / PCA ON EMBEDDINGS
# =========================
X_norm = l2_normalize_rows(X_text)
X_z = StandardScaler().fit_transform(X_norm)

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_SEED)
T = pca.fit_transform(X_z)

print("Explained variance (first 10):")
for i in range(min(10, N_COMPONENTS)):
    print(f"  PC{i+1}: {pca.explained_variance_ratio_[i]:.4f}")
print(f"  Cumulative ({N_COMPONENTS} PCs): {pca.explained_variance_ratio_.sum():.3f}")

# Save explained variance
var_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(N_COMPONENTS)],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
})
var_path = os.path.join(OUT_DIR, "svd_explained_variance.csv")
var_df.to_csv(var_path, index=False)
print("\nSaved:", var_path)

In [ ]:
# =========================
# EXEMPLAR TEXTS PER COMPONENT
# =========================
for k in range(N_COMPONENTS):
    pm_day[f"_svd_{k+1}"] = T[:, k]

# Within-person decomposition for top PCs
for k in range(min(5, N_COMPONENTS)):
    col = f"_svd_{k+1}"
    pm_day[f"{col}_between"] = pm_day.groupby(PID_COL)[col].transform("mean")
    pm_day[f"{col}_within"] = pm_day[col] - pm_day[f"{col}_between"]


def get_exemplars(df, score_col, direction="high", n=5, min_words=6):
    df = df[df["_wc"] >= min_words].copy()
    asc = (direction != "high")
    df = df.sort_values(score_col, ascending=asc).head(50)
    seen = set()
    chosen = []
    for _, row in df.iterrows():
        pid = str(row[PID_COL])
        if pid in seen:
            continue
        seen.add(pid)
        chosen.append({
            "component": score_col,
            "direction": direction,
            "score": float(row[score_col]),
            "word_count": int(row["_wc"]),
            "text": anonymize_text(str(row[TEXT_COL])),
        })
        if len(chosen) >= n:
            break
    return chosen


all_exemplars = []

# Global exemplars (overall PC scores)
for k in range(min(5, N_COMPONENTS)):
    col = f"_svd_{k+1}"
    all_exemplars += get_exemplars(pm_day, col, "high", N_EXEMPLARS)
    all_exemplars += get_exemplars(pm_day, col, "low", N_EXEMPLARS)

# Within-person exemplars (state-level deviations)
for k in range(min(3, N_COMPONENTS)):
    col_within = f"_svd_{k+1}_within"
    if col_within in pm_day.columns:
        all_exemplars += get_exemplars(pm_day, col_within, "high", N_EXEMPLARS)
        all_exemplars += get_exemplars(pm_day, col_within, "low", N_EXEMPLARS)

ex_df = pd.DataFrame(all_exemplars)
ex_path = os.path.join(OUT_DIR, "svd_component_exemplars.csv")
ex_df.to_csv(ex_path, index=False)

print(f"Saved {len(ex_df)} exemplars to: {ex_path}")
print("\nSample:")
for _, row in ex_df.head(6).iterrows():
    print(f"  [{row['component']} {row['direction']}] score={row['score']:.3f}")
    print(f"    {row['text'][:120]}...")

In [ ]:
# =========================
# WITHIN-PERSON EXTREME DAYS
# =========================
# For each participant, find their most unusual day (highest instability)
if "instability_cosdist" in pm_day.columns:
    extreme_days = []
    for pid, grp in pm_day.groupby(PID_COL):
        if len(grp) < 3:
            continue
        idx_max = grp["instability_cosdist"].idxmax()
        row = grp.loc[idx_max]
        if row["_wc"] >= MIN_WORDS:
            extreme_days.append({
                "participant": pid,
                "instability": float(row["instability_cosdist"]),
                "crisis": float(row[CRISIS_COL]) if pd.notna(row[CRISIS_COL]) else np.nan,
                "word_count": int(row["_wc"]),
                "text": anonymize_text(str(row[TEXT_COL])),
            })

    extreme_df = pd.DataFrame(extreme_days).sort_values("instability", ascending=False)
    extreme_path = os.path.join(OUT_DIR, "within_person_extreme_days.csv")
    extreme_df.to_csv(extreme_path, index=False)
    print(f"Saved {len(extreme_df)} extreme days to: {extreme_path}")
    print("\nTop 5 most unusual days:")
    for _, row in extreme_df.head(5).iterrows():
        print(f"  instability={row['instability']:.3f} crisis={row['crisis']}")
        print(f"    {row['text'][:120]}...")
else:
    print("instability_cosdist not found. Run notebook 03 first.")

In [ ]:
# =========================
# BERTOPIC (optional — skip gracefully if not installed)
# =========================
try:
    from bertopic import BERTopic
    from umap import UMAP

    print("BERTopic available. Running topic modeling...")

    texts = pm_day.loc[pm_day["_wc"] >= MIN_WORDS, TEXT_COL].tolist()
    X_topic = X_text[pm_day["_wc"] >= MIN_WORDS]

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=RANDOM_SEED)

    topic_model = BERTopic(
        umap_model=umap_model,
        nr_topics="auto",
        verbose=True,
    )

    topics, probs = topic_model.fit_transform(texts, X_topic)

    topic_info = topic_model.get_topic_info()
    topic_path = os.path.join(OUT_DIR, "bertopic_topics.csv")
    topic_info.to_csv(topic_path, index=False)
    print(f"\nSaved {len(topic_info)} topics to: {topic_path}")
    print(topic_info.head(10).to_string(index=False))

except ImportError:
    print("BERTopic not installed. To enable topic modeling:")
    print("  pip install bertopic umap-learn hdbscan")
    print("Skipping this analysis.")